# Recurrent Neural Network
### Recommender System

---
This synthetic recommendation dataset simulates user–item interaction sequences for teaching and experimenting with RNN-based next-item prediction models. It contains multiple users, each belonging to a preference cluster (e.g., action, comedy, drama), and their behavior is represented as short sequences of item IDs with smooth transitions within the preferred range. Each sequence is padded to a fixed length, and the goal is to predict the next item the user will interact with. The dataset is split into train and test CSV files, making it lightweight, privacy-safe, and easy to distribute without external downloads.

---

### Libraries

In [ ]:
import os
import sys

# Path
sys.path.append(os.path.abspath(".."))

# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

from numpy import array

import subprocess

from pathlib import Path
import shutil

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.models import *
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras import layers

# Model optimization
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.sparsity.keras import (
    prune,
    pruning_callbacks,
    pruning_schedule
)
from tensorflow_model_optimization.sparsity.keras import strip_pruning, prune_low_magnitude
from tensorflow_model_optimization.sparsity import keras as sparsity

from tensorflow.keras.preprocessing.sequence import pad_sequences

# QKeras (Quantization)
from qkeras import *

# Metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import shuffle

# Preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

# Dataset split
from sklearn.model_selection import train_test_split

# HLS4ML
import hls4ml

# Reproducibility
tf.random.set_seed(42)

# Clear sessions
K.clear_session()
tf.keras.backend.clear_session()

import sys
sys.path.append(os.path.abspath('../../'))
from common.notebook_utils.distillationClassKeras import Distiller
from common.notebook_utils.utils import  top_k_accuracy, mrr_at_k


os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

2026-03-30 22:47:28.507746: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 22:47:28.555044: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-30 22:47:29.165242: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


/tools/anaconda3/envs/neuralEnv/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


### Enable GPU 

In [2]:
#  GPU 
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'

import tensorflow as tf
print("GPUs: ", len(tf.config.experimental.list_physical_devices('GPU')))

import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

GPUs:  1


2026-03-30 22:47:30.215309: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-30 22:47:30.264122: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-30 22:47:30.264348: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

### Parameters

In [3]:
num_users=1000
num_items=100
seq_len=30
max_len=30

### Dataset

In [4]:

path = "../00.datasets/"

# Load train sequences and targets
X_train = pd.read_csv(path + "train_sequences.csv").values
y_train = pd.read_csv(path + "train_targets.csv")["target"].values

# Load test sequences and targets
X_test  = pd.read_csv(path + "test_sequences.csv").values
y_test  = pd.read_csv(path + "test_targets.csv")["target"].values

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)


Train: (11600, 30) (11600,)
Test : (2900, 30) (2900,)


### RNN-based model

In [5]:
# Hyperparameters
batch_size=64
epochs=32
lr = 3e-4
opt = Adam(lr)


2026-03-30 22:47:30.356590: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x3e7c0ce0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-03-30 22:47:30.356620: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): Host, Default Version
2026-03-30 22:47:30.356927: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-30 22:47:30.357163: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
20

In [6]:
# Define and create the model based on RNN for the recommendation task

# Input:  Sequence of item IDs (integer-encoded), length = max_len (default 30)
#         Represents the last N items interacted with by the user
# Body:   Embedding layer → GRU layer
#         ├─ Embedding: maps each item ID to a dense 8-dimensional vector
#         └─ GRU:       processes the sequence and outputs a single context vector
# Output: Softmax over all items (num_items) → probability of next item
#         The model predicts which item the user is most likely to interact with next

def create_rnn_recommender(num_items, max_len=30):

    inputs = keras.Input(shape=(max_len,), dtype="int32")

    x = layers.Embedding(num_items, 8, name = 'embedding_layer')(inputs)

    x = layers.GRU(16, return_sequences=False, name ="gru_layer")(x)

    outputs = layers.Dense(num_items, activation="softmax", name ="output_softmax")(x)

    model = keras.Model(inputs, outputs)
    
    model.compile(
        loss = "sparse_categorical_crossentropy",
        optimizer = opt,
        metrics = ["accuracy"]
    )
    return model

reco_model = create_rnn_recommender(num_items)
reco_model.summary()

2026-03-30 22:47:30.626544: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:30.627744: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:30.628606: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 30)]              0         
                                                                 
 embedding_layer (Embedding)  (None, 30, 8)            800       
                                                                 
 gru_layer (GRU)             (None, 16)                1248      
                                                                 
 output_softmax (Dense)      (None, 100)               1700      
                                                                 
Total params: 3,748
Trainable params: 3,748
Non-trainable params: 0
_________________________________________________________________


In [7]:
# Train the recommender model
history_recommender = reco_model.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)


Epoch 1/32


2026-03-30 22:47:30.979093: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:30.980305: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:30.981073: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

145/145 [==============================] - 3s 9ms/step - loss: 4.6009 - accuracy: 0.0388 - val_loss: 4.5933 - val_accuracy: 0.0565
Epoch 2/32
  1/145 [..............................] - ETA: 0s - loss: 4.5942 - accuracy: 0.0625

2026-03-30 22:47:33.204450: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:33.205716: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:33.206461: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

145/145 [==============================] - 1s 5ms/step - loss: 4.5199 - accuracy: 0.0325 - val_loss: 4.3683 - val_accuracy: 0.0276
Epoch 3/32
145/145 [==============================] - 0s 2ms/step - loss: 4.2871 - accuracy: 0.0276 - val_loss: 4.2209 - val_accuracy: 0.0353
Epoch 4/32
145/145 [==============================] - 0s 3ms/step - loss: 4.1811 - accuracy: 0.0325 - val_loss: 4.1478 - val_accuracy: 0.0297
Epoch 5/32
145/145 [==============================] - 0s 2ms/step - loss: 4.1185 - accuracy: 0.0406 - val_loss: 4.0985 - val_accuracy: 0.0315
Epoch 6/32
145/145 [==============================] - 0s 2ms/step - loss: 4.0728 - accuracy: 0.0427 - val_loss: 4.0596 - val_accuracy: 0.0336
Epoch 7/32
145/145 [==============================] - 0s 2ms/step - loss: 4.0348 - accuracy: 0.0437 - val_loss: 4.0246 - val_accuracy: 0.0397
Epoch 8/32
145/145 [==============================] - 0s 2ms/step - loss: 3.9981 - accuracy: 0.0496 - val_loss: 3.9888 - val_accuracy: 0.0509
Epoch 9/32
145/14

In [8]:
# Save the model
reco_model.save("./models/teacher_recomender.h5")

### Model performance   

In [9]:
preds_recommender = reco_model.predict(X_test, batch_size=64)

print("Recommender Top-1 :", top_k_accuracy(preds_recommender, y_test, 1))
print("Recommender Top-5 :", top_k_accuracy(preds_recommender, y_test, 5))
print("Recommender Top-10:", top_k_accuracy(preds_recommender, y_test, 10))

print("Recommender MRR@10:", mrr_at_k(preds_recommender, y_test, 10))

46/46 [==============================] - 0s 868us/step
Recommender Top-1 : 0.17206896551724138
Recommender Top-5 : 0.5893103448275862
Recommender Top-10: 0.8679310344827587
Recommender MRR@10: 0.34882402846195953


2026-03-30 22:47:43.760264: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:43.761418: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:43.762187: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

In [ ]:
def build_pruned_model(model, x_train_size, batch_size, epochs):

    # --- Pruning Schedule Setup ---
    steps_per_epoch    = int(np.ceil(x_train_size / batch_size))
    warmup_epochs      = 5                                              # Epochs before pruning begins
    finish_margin_ep   = 10                                             # Pruning stops N epochs before end
    finish_prune_epoch = max(warmup_epochs + 1, epochs - finish_margin_ep)

    begin_step = warmup_epochs      * steps_per_epoch   # Step where pruning starts
    end_step   = finish_prune_epoch * steps_per_epoch   # Step where pruning ends

    # PolynomialDecay: gradually increases sparsity from 0% to 40% between begin_step and end_step
    pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity = 0.0,          # No weights pruned at the start
        final_sparsity   = 0.4,          # 40% of weights zeroed out by end_step
        begin_step       = begin_step,
        end_step         = end_step,
        frequency        = steps_per_epoch   # Pruning mask updated once per epoch
    )

    # --- Apply Pruning to Dense Layers ---
    # Only standard Dense layers are wrapped; all other layer types are left unchanged
    def apply_pruning_to_layer(layer):
        if isinstance(layer, tf.keras.layers.Dense):
            return prune_low_magnitude(layer, pruning_schedule=pruning_schedule)
        return layer    # Leave all other layers (Dropout, Activation, etc.) unchanged

    # Clone the original model, applying pruning only to Dense layers
    pruned_model = tf.keras.models.clone_model(model, clone_function=apply_pruning_to_layer)

    return pruned_model

In [11]:
# Build the pruned version of the original recommendation model
# passing training size, batch size and epochs to configure the pruning schedule
precommender = build_pruned_model(reco_model, X_train.shape[0], batch_size, epochs)

# Compile the pruned model with:
# - optimizer: previously defined (e.g. Adam, SGD, etc.)
# - loss: sparse_categorical_crossentropy, suitable for integer-encoded class labels
# - metrics: tracking accuracy during training
precommender.compile(
    optimizer=opt,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Define training callbacks:
callbacks = [
    # Reduces learning rate when 'accuracy' stops improving for 5 consecutive epochs,
    # helping the model escape plateaus and fine-tune more carefully
    tf.keras.callbacks.ReduceLROnPlateau(monitor='accuracy', patience=5),
    
    # Required callback when using TF Model Optimization pruning,
    # updates the pruning masks at each training step based on the schedule
    tfmot.sparsity.keras.UpdatePruningStep()
]

# Train the pruned model:
# - Uses 80% of data for training, 20% for validation (validation_split=0.2)
# - Pruning is applied progressively during training via the UpdatePruningStep callback
history_precommender = precommender.fit(
    X_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,   # 20% of training data used for validation
    callbacks=callbacks
)

2026-03-30 22:47:44.083691: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:44.085220: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:44.086193: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

Epoch 1/32


2026-03-30 22:47:44.670187: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:44.671513: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:44.672475: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

145/145 [==============================] - 3s 4ms/step - loss: 2.7101 - accuracy: 0.1875 - val_loss: 2.7238 - val_accuracy: 0.1681 - lr: 3.0000e-04
Epoch 2/32
  1/145 [..............................] - ETA: 0s - loss: 2.7716 - accuracy: 0.1250

2026-03-30 22:47:46.983149: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:46.984182: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:46.984939: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

145/145 [==============================] - 0s 2ms/step - loss: 2.6787 - accuracy: 0.1859 - val_loss: 2.6926 - val_accuracy: 0.1716 - lr: 3.0000e-04
Epoch 3/32
145/145 [==============================] - 0s 2ms/step - loss: 2.6497 - accuracy: 0.1891 - val_loss: 2.6651 - val_accuracy: 0.1754 - lr: 3.0000e-04
Epoch 4/32
145/145 [==============================] - 0s 2ms/step - loss: 2.6227 - accuracy: 0.1940 - val_loss: 2.6385 - val_accuracy: 0.1780 - lr: 3.0000e-04
Epoch 5/32
145/145 [==============================] - 0s 2ms/step - loss: 2.5970 - accuracy: 0.1948 - val_loss: 2.6149 - val_accuracy: 0.1767 - lr: 3.0000e-04
Epoch 6/32
145/145 [==============================] - 0s 2ms/step - loss: 2.5733 - accuracy: 0.1930 - val_loss: 2.5916 - val_accuracy: 0.1759 - lr: 3.0000e-04
Epoch 7/32
145/145 [==============================] - 0s 2ms/step - loss: 2.5514 - accuracy: 0.1948 - val_loss: 2.5695 - val_accuracy: 0.1793 - lr: 3.0000e-04
Epoch 8/32
145/145 [==============================] - 0s 

In [ ]:
# strip_pruning removes all pruning-related wrapper layers and metadata that were added during training (e.g. pruning masks, step counters).
# The result is a clean, lightweight model where pruned weights remain as zeros, reducing model complexity without affecting the learned sparsity.
precommender_stripped = strip_pruning(precommender)


2026-03-30 22:47:57.689664: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:57.690922: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:57.691920: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

In [ ]:

# Save the stripped model to disk in HDF5 (.h5) format.
# This file contains the full model architecture, weights, and optimizer state, ready for inference or deployment 

precommender_stripped.save("./models/pmodel_stripped.h5")

In [14]:
for layer in precommender_stripped.layers:
    w = layer.get_weights()
    if len(w) > 0:
        zeros = np.sum(w[0] == 0)
        total = w[0].size
        print(layer.name, "sparsity:", zeros/total)


embedding_layer sparsity: 0.0
gru_layer sparsity: 0.0
output_softmax sparsity: 0.4


In [15]:
# Evaluate model performance 

preds_precommender = precommender_stripped.predict(X_test, batch_size=64)

print("PRecommender Top-1 :", top_k_accuracy(preds_precommender, y_test, 1))
print("PRecommender Top-5 :", top_k_accuracy(preds_precommender, y_test, 5))
print("PRecommender Top-10:", top_k_accuracy(preds_precommender, y_test, 10))

print("PRecommender MRR@10:", mrr_at_k(preds_precommender, y_test, 10))


46/46 [==============================] - 0s 855us/step
PRecommender Top-1 : 0.1910344827586207
PRecommender Top-5 : 0.646896551724138
PRecommender Top-10: 0.9279310344827586
PRecommender MRR@10: 0.3812205801860974


2026-03-30 22:47:57.926214: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_2_grad/concat/split_2/split_dim' with dtype int32
	 [[{{node gradients/split_2_grad/concat/split_2/split_dim}}]]
2026-03-30 22:47:57.927698: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'gradients/split_grad/concat/split/split_dim' with dtype int32
	 [[{{node gradients/split_grad/concat/split/split_dim}}]]
2026-03-30 22:47:57.928542: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You mus

With the training pipeline complete, the next notebook covers the HLS/ML workflow: from HLS4ML conversion to IP core generation.

**Next step:** Proceed to the next notebook *02.hls4ml/rnn-hls4ml.ipynb*.


---


This work was supported in part by the [AMD University Program](https://www.amd.com/en/corporate/university-program.html) 